# Analisis tracking + konkatenasi CSV

**Folder:** `Iter_2` (jalankan notebook dari sini).

1. Gabungkan semua `*.csv` sumber (kecuali file keluaran) + kolom `source_file`.
2. Ringkas **tracking loss** (`flag_lost`) dan **recently lost** (`flag_recently_lost`) — agregat & per file.
3. Ringkas **missing** (`missing_summary`).
4. **Imputasi** ringkas → `tracking_concat_imputed.csv`.
5. **One-hot** dari `concat.csv` (jika ada) → `concat_onehot.csv`.


In [ ]:
!pip install -q pandas numpy matplotlib


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd()
SKIP_PREFIX = (
    "concat_onehot",
    "tracking_concat_imputed",
    "concatenated",
)
SKIP_NAMES = {"concat.csv"}  # dipakai terpisah di bagian one-hot


def list_source_csvs(root: Path) -> list[Path]:
    out: list[Path] = []
    for p in sorted(root.glob("*.csv")):
        if p.name in SKIP_NAMES:
            continue
        if any(p.name.startswith(pref) for pref in SKIP_PREFIX):
            continue
        out.append(p)
    return out


paths = list_source_csvs(DATA_DIR)
if not paths:
    raise FileNotFoundError(f"Tidak ada CSV sumber di {DATA_DIR.resolve()}")

frames: list[pd.DataFrame] = []
for p in paths:
    d = pd.read_csv(p, low_memory=False)
    d["source_file"] = p.name
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
print("Gabungan:", df.shape, "| sumber:", [p.name for p in paths])
df.head(2)


In [ ]:
def missing_summary(d: pd.DataFrame, sample: int = 5) -> pd.DataFrame:
    rows = []
    for c in d.columns:
        s = d[c]
        nan_n = int(s.isna().sum())
        if s.dtype == object or pd.api.types.is_string_dtype(s):
            ss = s.astype(str)
            empty = (ss.str.strip() == "").sum()
            bad = ss.str.strip().str.lower().isin(["nan", "none", "n/a", ""]).sum() - empty
        else:
            empty, bad = 0, 0
        n = len(d)
        rows.append(
            {
                "column": c,
                "dtype": str(s.dtype),
                "nan_n": nan_n,
                "nan_pct": 100.0 * nan_n / max(n, 1),
                "empty_str": int(empty),
                "bad_token": int(bad),
            }
        )
    out = pd.DataFrame(rows).sort_values("nan_pct", ascending=False)
    return out.head(sample) if sample else out


miss = missing_summary(df, sample=None)
print(miss.head(25).to_string(index=False))

key = [
    c
    for c in [
        "flag_lost",
        "flag_recently_lost",
        "inlier_ratio",
        "reproj_error",
        "e_trans",
    ]
    if c in df.columns and "source_file" in df.columns
]
if key:
    g = df.groupby("source_file", sort=False)[key].apply(lambda x: 100 * x.isna().mean())
    print("\n% NaN per source_file (kolom kunci):")
    print(g.round(2).to_string())


In [ ]:
def tracking_stats(d: pd.DataFrame) -> None:
    n = len(d)
    if "flag_lost" in d.columns:
        tl = 100.0 * d["flag_lost"].fillna(0).astype(float).mean()
        print(f"Tracking loss (flag_lost=1): {tl:.2f}%")
    if "flag_recently_lost" in d.columns:
        rl = 100.0 * d["flag_recently_lost"].fillna(0).astype(float).mean()
        print(f"Recently lost (flag_recently_lost=1): {rl:.2f}%")
    if "tracking_state_name" in d.columns:
        print("\nDistribusi tracking_state_name:")
        print(d["tracking_state_name"].value_counts(dropna=False).head(20))
    if "source_file" in d.columns and "flag_lost" in d.columns:
        g = d.groupby("source_file", sort=False)
        rows = []
        for name, sub in g:
            r = {"source_file": name, "rows": len(sub)}
            r["pct_lost"] = 100.0 * sub["flag_lost"].fillna(0).astype(float).mean()
            if "flag_recently_lost" in sub.columns:
                r["pct_recent"] = 100.0 * sub["flag_recently_lost"].fillna(0).astype(float).mean()
            rows.append(r)
        print("\nPer source_file:")
        print(pd.DataFrame(rows).round(2).to_string(index=False))


tracking_stats(df)


### Imputasi ringkas → `tracking_concat_imputed.csv`


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
df_imputed = df.copy()
for c in num_cols:
    med = df_imputed[c].median()
    df_imputed[c] = df_imputed[c].fillna(med)

obj_cols = df_imputed.select_dtypes(include=["object"]).columns
for c in obj_cols:
    mode = df_imputed[c].mode(dropna=True)
    fill = mode.iloc[0] if len(mode) else ""
    df_imputed[c] = df_imputed[c].fillna(fill)

OUTPUT_CSV = DATA_DIR / "tracking_concat_imputed.csv"
df_imputed.to_csv(OUTPUT_CSV, index=False)
print("Tulis:", OUTPUT_CSV.resolve(), "| shape:", df_imputed.shape)


### One-hot dari `concat.csv` → `concat_onehot.csv`


In [ ]:
CONCAT_PATH = DATA_DIR / "concat.csv"
if not CONCAT_PATH.is_file():
    print("Tidak ada concat.csv — lewati one-hot.")
else:
    dc = pd.read_csv(CONCAT_PATH, low_memory=False)
    cols_oh = [
        c
        for c in dc.columns
        if dc[c].dtype == object or pd.api.types.is_string_dtype(dc[c])
    ]
    if "tracking_state" in dc.columns and dc["tracking_state"].nunique(dropna=True) <= 32:
        dc = dc.copy()
        dc["tracking_state_str"] = dc["tracking_state"].astype(str)
        cols_oh.append("tracking_state_str")
    cols_oh = list(dict.fromkeys([c for c in cols_oh if c in dc.columns]))
    dummies = pd.get_dummies(dc, columns=cols_oh, drop_first=False)
    out_oh = DATA_DIR / "concat_onehot.csv"
    dummies.to_csv(out_oh, index=False)
    print("Tulis:", out_oh.resolve(), "| shape:", dummies.shape)
